# spacetime on Google Colab

Installs the prebuilt `spacetime-continuum` wheel, imported as `spacetime` (it bundles a matching `gtsam`; do **not** `pip install gtsam` as well), then runs a small batch estimation and one of the demos headlessly.

In the local Docker test (`ci/test_fresh.sh`), the wheel comes from `/wheelhouse` and the examples from `/repo`. In Colab, the wheel is downloaded from the GitHub release and the examples are cloned from the repository.

In [ ]:
import glob, subprocess, sys, time

RELEASE = "v1.0.0-alpha"  # the GitHub release whose wheels to install
REPO = "https://github.com/utiasASRL/space_time_continuum"

local_wheels = sorted(glob.glob("/wheelhouse/spacetime_continuum-*-cp%d%d-*.whl" % sys.version_info[:2]))
if local_wheels:
    install = [local_wheels[-1]]                       # local Docker test
else:
    install = ["spacetime-continuum", "-f", f"{REPO}/releases/expanded_assets/{RELEASE}"]  # Colab
start = time.perf_counter()
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *install])
print(f"pip install: {time.perf_counter() - start:.1f} s")

In [ ]:
start = time.perf_counter()
import gtsam
import numpy as np
import spacetime as st
from spacetime import sensors
from spacetime.priors import wnoa
print(f"import spacetime: {time.perf_counter() - start:.2f} s")
print("spacetime", st.__file__)
print("gtsam    ", gtsam.__file__)

# gtsam and spacetime must share one libgtsam.
with open("/proc/self/maps") as maps:
    libgtsam = {line.split()[-1] for line in maps if "/libgtsam" in line}
print("libgtsam:", libgtsam)
assert len(libgtsam) == 1

## A bent robot in batch mode

A 0.2 m WNOA robot with a fixed base, whose tip is measured at a constant-curvature pose. The batch estimate should reach the measurement.

In [ ]:
length, curvature = 0.2, 3.0
base = gtsam.Pose3()
straight = np.array([0.0, 0.0, 0.0, 1.0, 0.0, 0.0])
tip = base.compose(gtsam.Pose3.Expmap(length * np.array([0.0, curvature, 0.0, 1.0, 0.0, 0.0])))

state = wnoa.ContinuumRobotState(length, 3)
reference = wnoa.NodeValues()
reference.pose(base)
reference.epsilon(straight)
prior = wnoa.Prior(state, reference, wnoa.PriorOptions())  # DefaultCovariances()
state.lock("pose", 0.0, base)

options = st.EstimatorOptions()
options.smootherLag = st.Estimator.BATCH_LAG
estimator = st.Estimator(options)
estimator.addModel(prior)

sensor = sensors.PoseSensor(prior)
covariance = np.diag([1e-4, 1e-4, 1e-4, 1e-6, 1e-6, 1e-6])
for k in range(6):
    sensor.addMeasurement(st.types.MeasurementBase(length, 0.1 * k, tip.matrix(), covariance,
                                                   np.ones((6, 1), dtype=np.int32)))
estimator.addSensor(sensor)

session_options = st.SessionOptions()
session_options.start_time, session_options.end_time, session_options.period = 0.0, 0.5, 0.1
session_options.real_time = False
session = st.Session(session_options)
start = time.perf_counter()
session.start(estimator)
session.join()
print(f"estimation: {time.perf_counter() - start:.2f} s")

estimate = prior.query(0.5, length, estimator.getCurrentValues()).pose()
print("measured tip :", np.round(tip.translation(), 4))
print("estimated tip:", np.round(estimate.translation(), 4))
assert estimate.equals(tip, 1e-2)

## A demo, headless

`python/examples/robot_joining.py`: two robots joined by a rigid link.

In [ ]:
import os, pathlib

examples = os.environ.get("SPACETIME_EXAMPLES")
if not examples:  # Colab: fetch the examples from the repository
    if not pathlib.Path("space_time_continuum").exists():
        subprocess.check_call(["git", "clone", "-q", "--depth", "1", REPO])
    examples = "space_time_continuum/python/examples"
output = pathlib.Path("/tmp/robot_joining_output")
result = subprocess.run([sys.executable, "robot_joining.py", "--headless", "--output", str(output)],
                        cwd=examples, capture_output=True, text=True)
print(result.stdout[-2000:], result.stderr[-2000:])
assert result.returncode == 0
print(sorted(p.name for p in output.iterdir()))